In [ ]:
import pandas as pd
import numpy as np 
import os
import json

In [ ]:
artifacts_path = "../exp_results/features_importance"

# Create the folder if it doesn't exist
os.makedirs(artifacts_path, exist_ok=True)

existing_experiments = []
if os.path.exists(artifacts_path):
    for folder_name in os.listdir(artifacts_path):
        folder_path = os.path.join(artifacts_path, folder_name)
        if os.path.isdir(folder_path) and folder_name.isdigit():
            existing_experiments.append(int(folder_name))


EXP_NUMBER = max(existing_experiments, default=0) + 1

#create the experience folder
os.makedirs(f"{artifacts_path}/{EXP_NUMBER}")

NUM_DATA_POINTS = 50000

print(f"Current experiment number: {EXP_NUMBER}")

**load the data**

In [ ]:
#read the json file
data_file_path = "../data/processed/multinli_1.0_train_cleaned.csv"
data = pd.read_csv(data_file_path, sep='µ')

#get a random sample of the data
data = data.sample(n = NUM_DATA_POINTS, random_state = 42)
data.head()

### Apply TF-IDF

In [ ]:
import spacy
import re
import unicodedata
from tqdm import tqdm

def clean_text(text):
    # Remove accented characters (optional)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    # Remove anything that is NOT a letter or space
    text = re.sub(r"[^a-zA-Z\s]", ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Lowercase the text
    text = text.lower()
    
    return text

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]  # disable unused components
)


def preprocess_texts(texts, batch_size=10000):
    cleaned = []

    # First, clean numbers/special chars
    texts = [clean_text(t) for t in texts]

    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts)):
        tokens = [
            token.lemma_.lower()
            for token in doc
        ]
        cleaned.append(" ".join(tokens))

    return cleaned

In [ ]:
#clean text sentence1 and sentence2
data['sentence1_cleaned'] = preprocess_texts(data['sentence1'].astype(str).tolist())
data['sentence2_cleaned'] = preprocess_texts(data['sentence2'].astype(str).tolist())

In [ ]:
#apply tf-idf, remove stop words, remove numbering
from sklearn.feature_extraction.text import TfidfVectorizer

all_sentences = data['sentence1_cleaned'].tolist() + data['sentence2_cleaned'].tolist()

# Fit on all text to get the same vocabulary
tfidf = TfidfVectorizer(min_df=0.001, ngram_range=(1, 2))
tfidf.fit(all_sentences)

# Transform each column separately using the same vectorizer
tfidf_s1_matrix = tfidf.transform(data['sentence1_cleaned']).toarray()
tfidf_s2_matrix = tfidf.transform(data['sentence2_cleaned']).toarray()

In [ ]:
#use the tfidf vectors as features (concatenate sentence1 and sentence2)
feature_names_s1 = [f"s1_{w}" for w in tfidf.get_feature_names_out()]
feature_names_s2 = [f"s2_{w}" for w in tfidf.get_feature_names_out()]
tf_idf_feature = feature_names_s1 + feature_names_s2

features = pd.DataFrame(
    np.hstack([tfidf_s1_matrix, tfidf_s2_matrix]),
    columns=tf_idf_feature
)

#add the features to the cleaned_data
data_tf_idf = pd.concat([data.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

#drop sentence1 and sentence2 but keep genre columns
data_tf_idf = data_tf_idf.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])
data_tf_idf.head()

In [ ]:
len(tf_idf_feature) / 2

### feature engineering

**features creation**

In [ ]:
#add feature length sentence1 and sentence2
data_tf_idf['s1_length'] = data_tf_idf['sentence1_cleaned'].apply(lambda x: len(str(x).split()))
data_tf_idf['s2_length'] = data_tf_idf['sentence2_cleaned'].apply(lambda x: len(str(x).split()))

#add feature to detect negation words
negation_words = set(['not', 'no', 'never', 'neither', 'nobody', 'nothing', 'nowhere', 'none', 'nor', 'cannot', "t", 'without', 'hardly', 'scarcely', 'barely', 'seldom', 'rarely'])

In [ ]:
#add feature who many negation words in sentence1 and sentence2
def count_negation_words(text):
    words = str(text).lower().split()
    return sum(1 for word in words if word in negation_words)

data_tf_idf['s1_negation_count'] = data_tf_idf['sentence1_cleaned'].apply(count_negation_words)
data_tf_idf['s2_negation_count'] = data_tf_idf['sentence2_cleaned'].apply(count_negation_words)

#add percentage of shared words between sentence1 and sentence2
def shared_word_percentage(row):
    s1_words = set(str(row['sentence1_cleaned']).split())
    s2_words = set(str(row['sentence2_cleaned']).split())
    if len(s1_words) == 0 or len(s2_words) == 0:
        return 0.0
    shared_words = s1_words.intersection(s2_words)
    total_words = s1_words.union(s2_words)
    return len(shared_words) / len(total_words)

data_tf_idf['shared_word_percentage_sentence1_sentence2'] = data_tf_idf.apply(shared_word_percentage, axis=1)

In [ ]:
# add feature to detect antonyms between sentence1 and sentence2
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def count_antonyms(row):
    s1_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence1_cleaned']).split()]
    s2_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence2_cleaned']).split()]
    
    antonym_count = 0
    
    for word in s1_words:
        antonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                for ant in lemma.antonyms():  # include all antonyms
                    antonyms.add(lemmatizer.lemmatize(ant.name().lower()))
        
        # Increment if any antonym exists in S2
        if any(ant in s2_words for ant in antonyms):
            antonym_count += 1
            
    return antonym_count

data_tf_idf['antonym_count'] = data_tf_idf.apply(count_antonyms, axis=1)

In [ ]:
#add feature consine similarity between sentence1 and sentence2
from sklearn.metrics.pairwise import cosine_similarity

def compute_cosine_similarity(row):
    s1_vector = row[tf_idf_feature[:len(tf_idf_feature)//2]].values.reshape(1, -1)
    s2_vector = row[tf_idf_feature[len(tf_idf_feature)//2:]].values.reshape(1, -1)
    
    return cosine_similarity(s1_vector, s2_vector)[0][0]

data_tf_idf['cosine_similarity'] = data_tf_idf.apply(compute_cosine_similarity, axis=1)

**feature combinaison**

In [ ]:
data_tf_idf["ratio_length_sentence1_sentence2"] = data_tf_idf["s1_length"] / (data_tf_idf["s2_length"] + 1) 
data_tf_idf['ratio_negation_count_sentence1_sentence2'] = data_tf_idf['s1_negation_count'] / (data_tf_idf['s2_negation_count'] + 1)
data_tf_idf['antonym_count_ratio_sentence1_sentence2'] = data_tf_idf['antonym_count'] / (data_tf_idf["s1_length"] + data_tf_idf["s2_length"])

In [ ]:
# #correlation matric between the features we calculated

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
correlation_matrix = data_tf_idf[[
    "s1_length",
    "s2_length",
    "ratio_length_sentence1_sentence2",
    "s1_negation_count",
    "s2_negation_count",
    "ratio_negation_count_sentence1_sentence2",
    "shared_word_percentage_sentence1_sentence2",
    "antonym_count",
    "antonym_count_ratio_sentence1_sentence2",
    "cosine_similarity"
]].corr()

sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Correlation Matrix of Engineered Features")
plt.show()

In [ ]:
data_tf_idf.drop(columns=['sentence1_cleaned', 'sentence2_cleaned'], inplace=True)

**feature importance**

In [ ]:
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def stepwise_with_base_features(
    X_train_full, X_val_full,
    y_train_enc, y_val_enc,
    base_features, candidate_features,
    model, baseline_score, tol = 10e-3
):
    selected_features = []
    remaining_features = candidate_features.copy()
    best_score = baseline_score

    feature_history = []

    while True:
        changed = False

        # ----- FORWARD STEP -----
        scores_with_candidates = []
        for feature in remaining_features:
            features_to_test = base_features + selected_features + [feature]
            model.fit(X_train_full[features_to_test], y_train_enc)
            y_pred_enc = model.predict(X_val_full[features_to_test])
            score = accuracy_score(y_val_enc, y_pred_enc)

            scores_with_candidates.append((score, feature))
            print(f"testing add {feature:40s} => acc {score:.4f}")

        if scores_with_candidates:
            score, best_feature = max(scores_with_candidates)
            if score > best_score + tol:
                delta_prev = score - best_score
                delta_base = score - baseline_score

                selected_features.append(best_feature)
                remaining_features.remove(best_feature)
                best_score = score
                changed = True

                feature_history.append({
                    "feature": best_feature,
                    "accuracy": score,
                    "delta_vs_previous": delta_prev,
                    "delta_vs_baseline": delta_base
                })

                print(
                    f"ADD    ➜ {best_feature:40s} | "
                    f"acc={score:.4f} | "
                    f"Δprev={delta_prev:+.4f} | "
                    f"Δbase={delta_base:+.4f}"
                )

        # ----- BACKWARD STEP -----
        while selected_features:
            scores_after_removal = []
            for feature in selected_features:
                features_to_test = base_features + [
                    f for f in selected_features if f != feature
                ]
                model.fit(X_train_full[features_to_test], y_train_enc)
                y_pred_enc = model.predict(X_val_full[features_to_test])
                score = accuracy_score(y_val_enc, y_pred_enc)

                scores_after_removal.append((score, feature))
                print(f"testing remove {feature:37s} => acc {score:.4f}")

            score, worst_feature = max(scores_after_removal)

            if score >= best_score - tol:
                selected_features.remove(worst_feature)
                remaining_features.append(worst_feature)
                best_score = score
                changed = True
                print(f"REMOVE ➜ {worst_feature:40s} | acc={best_score:.4f}")
            else:
                break

        if not changed:
            break

    return selected_features, best_score, feature_history

In [ ]:
extra_features = [
    "s1_length",
    "s2_length",
    "ratio_length_sentence1_sentence2",
    "s1_negation_count",
    "s2_negation_count",
    "ratio_negation_count_sentence1_sentence2",
    "shared_word_percentage_sentence1_sentence2",
    "antonym_count",
    "antonym_count_ratio_sentence1_sentence2",
    "cosine_similarity"
]

tfidf_features = [
    c for c in data_tf_idf.columns
    if c not in extra_features + ["gold_label"]
]

y = data_tf_idf["gold_label"]

test_size = 0.2

X_train, X_val, y_train, y_val = train_test_split(
    data_tf_idf.drop(columns=["gold_label"]),
    y,
    test_size=test_size,
    random_state=42
)


In [ ]:
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_model.fit(X_train[tfidf_features], y_train_enc)

y_val_pred_enc = xgb_model.predict(X_val[tfidf_features])
baseline_accuracy = accuracy_score(y_val_enc, y_val_pred_enc)

print(f"Baseline TF-IDF validation accuracy: {baseline_accuracy:.4f}")

selected_features, final_accuracy, feature_history = stepwise_with_base_features(
    X_train_full=X_train,
    X_val_full=X_val,
    y_train_enc=y_train_enc,
    y_val_enc=y_val_enc,
    base_features=tfidf_features,
    candidate_features=extra_features,
    model=xgb_model,
    baseline_score=baseline_accuracy,
    tol= (4 / (NUM_DATA_POINTS * test_size))
)


results = {
    "baseline_accuracy_tfidf": baseline_accuracy,
    "final_accuracy": final_accuracy,
    "selected_features": selected_features,
    "feature_contributions": feature_history
}

OUTPUT_PATH = f"../exp_results/features_importance/{EXP_NUMBER}/feature_engineering_importance_results.json"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=2)


print("\nFinal selected extra features:")
for f in selected_features:
    print("-", f)

print(f"\nFinal validation accuracy (with TF-IDF + extras): {final_accuracy:.4f}")
print(f"\nResults written to {OUTPUT_PATH}")